In [12]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
import os
from torchvision import transforms
from PIL import Image
import numpy as np
from torchviz import make_dot
from tqdm import tqdm

In [ ]:
DATA_DIR = "../../data"
DATASET_DIR = f"{DATA_DIR}/processed"

In [ ]:
LE = LabelEncoder()

from snntorch import spikegen

class CustomDatasetSpike(Dataset):
    def __init__(self, dataframe, image_dir, transform=None, num_steps=4):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform = transform
        self.num_steps = num_steps

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.dataframe.iloc[idx]["Image"])
        with Image.open(image_path) as img:
            if self.transform:
                img = self.transform(img)
            label = torch.as_tensor(self.dataframe.iloc[idx]["Label"], dtype=torch.long)
            img = spikegen.rate(img, num_steps=self.num_steps, gain=1)
            return img, label


class CustomDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.dataframe.iloc[idx]["Image"])
        with Image.open(image_path) as img:
            if self.transform:
                img = self.transform(img)
            label = torch.as_tensor(self.dataframe.iloc[idx]["Label"], dtype=torch.long)
            return img, label

In [ ]:
test_df = pd.read_csv(f"{DATASET_DIR}/test1.csv")        

In [ ]:
BATCH_SIZE = 40

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor()
])

In [ ]:
test_data_encoded = test_df.copy()
test_data_encoded["Label"] = LE.fit_transform(test_data_encoded["Label"])

test_data_sample = test_data_encoded.groupby("Label").sample(2000, random_state=1)

test_dataset_sample = CustomDataset(test_data_sample, f"{DATASET_DIR}/images", transform=transform)
test_data_loader_sample = DataLoader(test_dataset_sample, batch_size=BATCH_SIZE, shuffle=False)

test_dataset_sample_spike = CustomDatasetSpike(test_data_sample, f"{DATASET_DIR}/images", transform=transform, num_steps=4)
test_data_loader_sample_spike = DataLoader(test_dataset_sample_spike, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
def visualize_model_graph(model, input, save_path=None):
    model.eval()
    with torch.no_grad():
        out = model(input)
    
    dot = make_dot(out, params=dict(model.named_parameters()))
    if save_path:
        dot.render(save_path, format="png")
    
    return dot

In [ ]:
def inference(model, dataloader, device):
    with torch.no_grad():
        


In [ ]:
class BinaryCNN3(nn.Module):
    def __init__(self, num_classes=2):
        super(BinaryCNN3, self).__init__()
        
        # Feature extraction layers
        self.conv1 = nn.Sequential(
            # First block: 32x32 -> 16x16
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.3),
            
        )

        self.conv2 = nn.Sequential(
            # Second block: 16x16 -> 8x8
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.3)
        )

        
        # Binary classifier
        self.classifier = nn.Sequential(
            nn.Linear(32 * 8 * 8 , 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, num_classes)
        )
    
    def forward(self, x):
        x = self.conv1(x)
        # print(x.shape)
        x = self.conv2(x)
        # print(x.shape)
        x = torch.flatten(x, 1)
        # print(x.shape)
        return self.classifier(x)

In [ ]:
IMAGE_DIR_PATH = "../../reports/figures"
MODEL_PATH = "../../models/checkpoints"

model = BinaryCNN3(num_classes=2)
checkpoint = torch.load(f"{MODEL_PATH}/cnn/bcnn3_crossentropy_cosine.pt", weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])

# Visualize the model graph
input = torch.randn(1, 1, 32, 32)
dot = visualize_model_graph(model, input, save_path=f"{IMAGE_DIR_PATH}/bcnn3_model_graph.png")